In [0]:
# UDF - User Defined Functions, custom functions written in scala,java, python 
# usedin spakr sql, spark dataframe
# python - slow
# scala, java - fast

# This example uses hardcoded dataset

In [0]:
# Databricks notebook source
# UDF - User Defined Functions
# useful to extend spark sql functions with custom code

power = lambda n : n * n

from pyspark.sql.functions import udf 
from pyspark.sql.types import LongType
# create udf with return type
powerUdf = udf(power, LongType())

# we must register udf in spark session
# udf too private within spark session, udf registered in spark session not avaialble in another spark session
# "power" is udf function name, can be used sql
spark.udf.register("power", powerUdf)


<function __main__.<lambda>(n)>

In [0]:

# power is udf function
spark.sql("SELECT power(5)").show()

+--------+
|power(5)|
+--------+
|      25|
+--------+



In [0]:
# Databricks notebook source
# Databricks notebook source
orders = [ 
          # (product_id, product_name, brand_id, price, qty, discount, taxp)  
         (1, 'iPhone', 100, 1000, 2, 5, 18),
         (2, 'Galaxy', 200, 800, 1, 8, 22),

]
 


orderDf = spark.createDataFrame(data=orders, schema=["product_id", "product_name", "brand_id", "price", "qty", "discount", "taxp"])
orderDf.show()

+----------+------------+--------+-----+---+--------+----+
|product_id|product_name|brand_id|price|qty|discount|taxp|
+----------+------------+--------+-----+---+--------+----+
|         1|      iPhone|     100| 1000|  2|       5|  18|
|         2|      Galaxy|     200|  800|  1|       8|  22|
+----------+------------+--------+-----+---+--------+----+



In [0]:
# UDF to calculate amount
# amount = ( price * qty ) * apply discount * taxp

def calculateAmount(price, qty, discount, taxp):
    a = price * qty
    a = a - (a * discount/100) # discounted price
    amount = a + a * taxp / 100 # with tax
    print ("amount is" , amount) 
    return amount

print(calculateAmount(1000, 2, 5, 18))

amount is 2242.0
2242.0


In [0]:

from pyspark.sql.types import DoubleType
# udf function
calculate = udf(calculateAmount, DoubleType())
# "calculate" is used in spark sql SELECT calculate(...)
spark.udf.register("calculate", calculate)

<function __main__.calculateAmount(price, qty, discount, taxp)>

In [0]:
# use udf in data frame
from pyspark.sql.functions import col
df = orderDf.withColumn("amount", calculate( col("price"), col("qty"), col("discount"), col("taxp")))
df.printSchema()
df.show()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_id: long (nullable = true)
 |-- price: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- taxp: long (nullable = true)
 |-- amount: double (nullable = true)

+----------+------------+--------+-----+---+--------+----+------+
|product_id|product_name|brand_id|price|qty|discount|taxp|amount|
+----------+------------+--------+-----+---+--------+----+------+
|         1|      iPhone|     100| 1000|  2|       5|  18|2242.0|
|         2|      Galaxy|     200|  800|  1|       8|  22|897.92|
+----------+------------+--------+-----+---+--------+----+------+



In [0]:
# create a temp table/view
orderDf.createOrReplaceTempView("orders")

In [0]:
## now apply udf on spark sql

df = spark.sql("SELECT *, calculate(price, qty, discount, taxp) as amount from orders")
df.printSchema()
df.show()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_id: long (nullable = true)
 |-- price: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- taxp: long (nullable = true)
 |-- amount: double (nullable = true)

+----------+------------+--------+-----+---+--------+----+------+
|product_id|product_name|brand_id|price|qty|discount|taxp|amount|
+----------+------------+--------+-----+---+--------+----+------+
|         1|      iPhone|     100| 1000|  2|       5|  18|2242.0|
|         2|      Galaxy|     200|  800|  1|       8|  22|897.92|
+----------+------------+--------+-----+---+--------+----+------+



In [0]:
%scala
// Will not work on free edition/serverless 

// Define Scala function
def calculateAmount(price: Double, qty: Int, discount: Double, taxp: Double): Double = {
  var a = price * qty
  a = a - (a * discount / 100)  // discounted price
  val amount = a + (a * taxp / 100) // with tax
  println(s"amount is $amount")
  amount
}

// Test the function
println(calculateAmount(1000, 2, 5, 18))

// Register as UDF in Spark
import org.apache.spark.sql.functions.udf

val calculateAmountUDF = udf((price: Double, qty: Int, discount: Double, taxp: Double) => {
  var a = price * qty
  a = a - (a * discount / 100)
  val amount = a + (a * taxp / 100)
  amount
})

spark.udf.register("calculate_scala_udf", calculateAmountUDF)

In [0]:
df = spark.sql("SELECT *, calculate_scala_udf(price, qty, discount, taxp) as amount from orders")
df.printSchema()
df.show()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand_id: long (nullable = true)
 |-- price: long (nullable = true)
 |-- qty: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- taxp: long (nullable = true)
 |-- amount: double (nullable = true)

+----------+------------+--------+-----+---+--------+----+------+
|product_id|product_name|brand_id|price|qty|discount|taxp|amount|
+----------+------------+--------+-----+---+--------+----+------+
|         1|      iPhone|     100| 1000|  2|       5|  18|2242.0|
|         2|      Galaxy|     200|  800|  1|       8|  22|897.92|
+----------+------------+--------+-----+---+--------+----+------+

